# Notebook 04: Mechanistic Interpretability -- Activation Patching & Circuit Discovery

**Prerequisites**: Transformer circuits (nb01), superposition (nb02)

Activation patching answers the question: **does this component actually matter?** This notebook covers the core causal methods for finding out -- activation patching, attribution patching, and circuit discovery on the IOI task.

## Section 1: Causal Intervention Methods

How do you determine which model components are responsible for a specific behavior? You use **causal interventions** -- modify internal activations and measure the effect on the output.

Three main approaches:
1. **Activation patching** (interchange intervention): Replace one activation with another from a different input
2. **Zero/mean ablation**: Replace activations with zero or their mean value
3. **Noising**: Add noise to activations to disrupt them

Here's the core protocol:
1. Run model on **clean** input (correct behavior), cache activations
2. Run model on **corrupted** input (wrong behavior)
3. **Patch** a clean activation into the corrupted run at a specific component
4. If output recovers --> that component is causally important for the behavior

## Section 2: Activation Patching on the IOI Task

The **Indirect Object Identification (IOI)** task is the canonical test case ([Wang et al., 2022](https://arxiv.org/abs/2211.00593)). Given "When Mary and John went to the store, John gave a drink to", the model should predict "Mary" (the indirect object).

We corrupt by replacing "Mary" with another name, then patch to find which components are needed to get "Mary" back.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer
from transformer_lens import utils

model = HookedTransformer.from_pretrained("gpt2-small")
model.eval()

# Clean and corrupted prompts
clean_prompt = "When Mary and John went to the store, John gave a drink to"
corrupted_prompt = "When Alice and John went to the store, John gave a drink to"

# Get token IDs for the answer
mary_token = model.to_single_token(" Mary")
alice_token = model.to_single_token(" Alice")

# Run clean and corrupted
clean_logits, clean_cache = model.run_with_cache(clean_prompt)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_prompt)

# Check model gets it right
clean_answer_logit = clean_logits[0, -1, mary_token].item()
corrupted_answer_logit = corrupted_logits[0, -1, mary_token].item()

print(f"Clean logit for ' Mary': {clean_answer_logit:.2f}")
print(f"Corrupted logit for ' Mary': {corrupted_answer_logit:.2f}")
print(f"Logit difference: {clean_answer_logit - corrupted_answer_logit:.2f}")

# Metric: logit difference (Mary - Alice)
clean_logit_diff = (clean_logits[0, -1, mary_token] - clean_logits[0, -1, alice_token]).item()
corrupted_logit_diff = (corrupted_logits[0, -1, mary_token] - corrupted_logits[0, -1, alice_token]).item()
print(f"\nClean logit diff (Mary - Alice): {clean_logit_diff:.2f}")
print(f"Corrupted logit diff (Mary - Alice): {corrupted_logit_diff:.2f}")

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

def patch_residual_stream(corrupted_activation, hook, pos, clean_cache, layer):
    """Replace corrupted residual stream with clean at a specific position."""
    corrupted_activation[0, pos, :] = clean_cache[f"blocks.{layer}.hook_resid_post"][0, pos, :]
    return corrupted_activation

n_layers = model.cfg.n_layers
clean_tokens = model.to_tokens(clean_prompt)
n_positions = clean_tokens.shape[1]
token_labels = [model.tokenizer.decode(t) for t in clean_tokens[0]]

patching_results = torch.zeros(n_layers, n_positions)

for layer in range(n_layers):
    for pos in range(n_positions):
        # Run corrupted input with clean patch at (layer, pos)
        hook_fn = lambda act, hook, p=pos, l=layer: patch_residual_stream(
            act, hook, p, clean_cache, l
        )
        patched_logits = model.run_with_hooks(
            corrupted_prompt,
            fwd_hooks=[(f"blocks.{layer}.hook_resid_post", hook_fn)]
        )
        patched_logit_diff = (patched_logits[0, -1, mary_token] - patched_logits[0, -1, alice_token]).item()
        
        # Normalize: 0 = no recovery (corrupted), 1 = full recovery (clean)
        patching_results[layer, pos] = (patched_logit_diff - corrupted_logit_diff) / (clean_logit_diff - corrupted_logit_diff)

# Interactive plotly heatmap
fig = px.imshow(
    patching_results.detach().numpy(),
    x=token_labels,
    y=list(range(n_layers)),
    labels=dict(x="Token Position", y="Layer", color="Recovery Fraction"),
    color_continuous_scale="RdBu",
    zmin=-0.5,
    zmax=1.0,
    aspect="auto",
    title="Activation Patching: Residual Stream (IOI Task)",
)
fig.update_traces(
    hovertemplate="Token: %{x}<br>Layer: %{y}<br>Recovery Fraction: %{z:.4f}<extra></extra>"
)
fig.update_layout(height=600, width=900)
fig.show()

## Section 3: Attention Head Patching

We can also patch individual attention head outputs to find which heads matter most.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

head_patching_results = torch.zeros(n_layers, model.cfg.n_heads)

for layer in range(n_layers):
    for head in range(model.cfg.n_heads):
        def patch_head_output(activation, hook, l=layer, h=head):
            # NOTE: This patches all sequence positions at once, which is a simplification.
            # Position-specific patching (e.g., only at the last token or IO position)
            # would give more precise results about where each head matters.
            activation[0, :, h, :] = clean_cache[f"blocks.{l}.attn.hook_result"][0, :, h, :]
            return activation
        
        patched_logits = model.run_with_hooks(
            corrupted_prompt,
            fwd_hooks=[(f"blocks.{layer}.attn.hook_result", patch_head_output)]
        )
        patched_logit_diff = (patched_logits[0, -1, mary_token] - patched_logits[0, -1, alice_token]).item()
        head_patching_results[layer, head] = (patched_logit_diff - corrupted_logit_diff) / (clean_logit_diff - corrupted_logit_diff)

layer_labels = [f"L{i}" for i in range(n_layers)]
head_labels = [f"H{i}" for i in range(model.cfg.n_heads)]

fig = px.imshow(
    head_patching_results.detach().numpy(),
    x=head_labels,
    y=layer_labels,
    labels=dict(x="Head", y="Layer", color="Recovery Fraction"),
    color_continuous_scale="RdBu",
    zmin=-0.3,
    zmax=0.3,
    aspect="auto",
    title="Attention Head Patching (IOI Task)",
)
fig.update_traces(
    hovertemplate="Layer: %{y}<br>Head: %{x}<br>Recovery Fraction: %{z:.4f}<extra></extra>"
)
fig.update_layout(height=600, width=900)
fig.show()

# Print most important heads
flat_results = head_patching_results.flatten()
top_positive = flat_results.argsort(descending=True)[:5]
top_negative = flat_results.argsort()[:5]

print("Top 5 heads that HELP (patching recovers performance):")
for idx in top_positive:
    l = idx.item() // model.cfg.n_heads
    h = idx.item() % model.cfg.n_heads
    print(f"  L{l}H{h}: {head_patching_results[l, h]:.3f}")

print("\nTop 5 heads that HURT (patching makes it worse — backup/inhibition heads):")
for idx in top_negative:
    l = idx.item() // model.cfg.n_heads
    h = idx.item() % model.cfg.n_heads
    print(f"  L{l}H{h}: {head_patching_results[l, h]:.3f}")

## Section 4: Attribution Patching (Scaling Up)

Activation patching requires O(components) forward passes. For large models, that's expensive. **Attribution patching** ([Neel Nanda, 2023](https://www.neelnanda.io/mechanistic-interpretability/attribution-patching)) approximates the patching effect using gradients:

Effect ~ (clean_activation - corrupted_activation) * gradient_of_metric_wrt_activation

This is a first-order Taylor approximation -- one forward + backward pass instead of thousands of forward passes. Much faster, but approximate.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Attribution patching using gradients
corrupted_tokens = model.to_tokens(corrupted_prompt)

# NOTE: The forward pass below is redundant (dead code) -- the run_with_hooks call
# further down repeats this computation with gradient hooks attached.
# corrupted_logits_grad = model(corrupted_tokens)
# logit_diff = corrupted_logits_grad[0, -1, mary_token] - corrupted_logits_grad[0, -1, alice_token]

# We need gradients of the metric w.r.t. each residual stream position
# Use hooks to collect gradients
grad_cache = {}

def save_grad_hook(activation, hook):
    activation.retain_grad()
    grad_cache[hook.name] = activation
    return activation

# Register hooks
hooks = []
for layer in range(n_layers):
    hooks.append((f"blocks.{layer}.hook_resid_post", save_grad_hook))

corrupted_logits_grad = model.run_with_hooks(corrupted_prompt, fwd_hooks=hooks)
logit_diff = corrupted_logits_grad[0, -1, mary_token] - corrupted_logits_grad[0, -1, alice_token]
logit_diff.backward()

# Compute attribution scores
attribution_scores = torch.zeros(n_layers, n_positions)
for layer in range(n_layers):
    act = grad_cache[f"blocks.{layer}.hook_resid_post"]
    grad = act.grad[0]  # (seq_len, d_model)
    clean_act = clean_cache[f"blocks.{layer}.hook_resid_post"][0]
    corrupted_act = act.detach()[0]
    
    # Attribution = (clean - corrupted) * grad, summed over d_model
    diff = clean_act - corrupted_act
    attribution_scores[layer] = (diff * grad).sum(dim=-1).detach().cpu()

abs_max = attribution_scores.abs().max().item()

fig = px.imshow(
    attribution_scores.numpy(),
    x=token_labels,
    y=list(range(n_layers)),
    labels=dict(x="Token Position", y="Layer", color="Attribution Score"),
    color_continuous_scale="RdBu",
    zmin=-abs_max,
    zmax=abs_max,
    aspect="auto",
    title="Attribution Patching (gradient approximation)",
)
fig.update_traces(
    hovertemplate="Token: %{x}<br>Layer: %{y}<br>Attribution Score: %{z:.4f}<extra></extra>"
)
fig.update_layout(height=600, width=900)
fig.show()

## Section 5: From Patching to Circuits

Activation patching identifies important components. To find a **circuit**, you need to:

1. Identify the minimal set of components needed for the behavior
2. Understand how they compose (information flow)
3. Verify by ablating the circuit and checking that behavior is preserved

The IOI circuit ([Wang et al., 2022](https://arxiv.org/abs/2211.00593)) identified these component types:
- **Name mover heads**: Copy the indirect object name to the output
- **Backup name movers**: Redundant name movers (the model has redundancy!)
- **Negative name movers**: Boost the subject (S) token in the logits, acting opposite to name movers
- **S-inhibition heads**: Detect the repeated subject and inhibit it
- **Duplicate token heads**: Detect repeated tokens in context
- **Previous token heads**: Attend to the previous position (enable induction)

This circuit spans ~26 heads across 12 layers -- a significant fraction of the model.

**Key lesson**: Real circuits are messier than toy examples. Models have redundancy, backup mechanisms, and distributed computation.

---
### Running Example: IOI — Activation Patching (This Notebook)

This notebook's main analysis **IS** the running example. The entire IOI circuit we have identified here — name mover heads, S-inhibition heads, duplicate token heads, and backup name movers — will be examined from other angles in the other notebooks in this series:

- **Notebook 01**: Attention patterns of the name mover heads
- **Notebook 03**: SAE features active at the prediction position
- **Notebook 05**: At which layer the model first starts predicting "Mary"
- **Notebook 06**: Linear probes that detect which name the model will output
- **Notebook 07**: Steering the model's name prediction with representation engineering
- **Notebook 08**: Attribution graph showing information flow through the IOI circuit
- **Notebook 09**: A complete, compact IOI investigation script

---
## Exercises

### Exercise 1: Patch a Different Component

Extend the IOI activation patching to patch individual attention heads (not just full layers). Which specific heads matter most for the IOI task? Create a heatmap of (layer, head) showing the logit difference change when each individual head is patched.

The notebook already does this in Section 3, but your task is to reproduce it independently and identify the top 3 most important heads. For each, explain its likely functional role (name mover, S-inhibition, duplicate token, etc.) based on its layer and the sign of its patching effect.

<details>
<summary>Hint</summary>

Use `hook_result` at each `blocks.{layer}.attn.hook_result` to patch individual head outputs. The hook_result tensor has shape `(batch, pos, n_heads, d_model)`. To patch only head `h`, replace `activation[0, :, h, :] = clean_cache[f"blocks.{l}.attn.hook_result"][0, :, h, :]`. Positive patching effect = the head helps (name mover), negative = the head hurts when restored (backup/inhibition head).

</details>

In [ ]:
# Exercise 1: Patch Individual Attention Heads

# Setup — reuse variables from earlier in notebook
clean_prompt = "When Mary and John went to the store, John gave a drink to"
corrupted_prompt = "When Alice and John went to the store, John gave a drink to"

# 1. Run clean and corrupted prompts, cache activations
clean_logits, clean_cache = model.run_with_cache(clean_prompt)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_prompt)

# 2. Get token IDs and baseline logit differences
mary_token = model.to_single_token(" Mary")
alice_token = model.to_single_token(" Alice")
clean_logit_diff = (clean_logits[0, -1, mary_token] - clean_logits[0, -1, alice_token]).item()
corrupted_logit_diff = (corrupted_logits[0, -1, mary_token] - corrupted_logits[0, -1, alice_token]).item()

# 3. For each (layer, head), patch that head's output from clean into corrupted
# TODO: Try patching at specific positions only (e.g., last position) instead of all positions!
my_head_results = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)
for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):
        def patch_head(activation, hook, l=layer, h=head):
            activation[0, :, h, :] = clean_cache[f"blocks.{l}.attn.hook_result"][0, :, h, :]
            return activation
        patched_logits = model.run_with_hooks(
            corrupted_prompt,
            fwd_hooks=[(f"blocks.{layer}.attn.hook_result", patch_head)]
        )
        patched_diff = (patched_logits[0, -1, mary_token] - patched_logits[0, -1, alice_token]).item()
        my_head_results[layer, head] = (patched_diff - corrupted_logit_diff) / (clean_logit_diff - corrupted_logit_diff)

# 4. Create heatmap
plt.figure(figsize=(14, 8))
plt.imshow(my_head_results.numpy(), cmap="RdBu", vmin=-0.3, vmax=0.3, aspect="auto")
plt.colorbar(label="Fraction of logit diff recovered")
plt.xlabel("Head")
plt.ylabel("Layer")
plt.title("Exercise: Attention Head Patching (IOI Task)")
plt.tight_layout()
plt.show()

# 5. Print top 3 positive and top 3 negative heads, and hypothesize their roles
flat = my_head_results.flatten()
top3_pos = flat.argsort(descending=True)[:3]
top3_neg = flat.argsort()[:3]

print("Top 3 heads that HELP (likely name mover heads):")
for idx in top3_pos:
    l, h = idx.item() // model.cfg.n_heads, idx.item() % model.cfg.n_heads
    print(f"  L{l}H{h}: {my_head_results[l, h]:.3f}")

print("\nTop 3 heads that HURT (likely backup/inhibition heads):")
for idx in top3_neg:
    l, h = idx.item() // model.cfg.n_heads, idx.item() % model.cfg.n_heads
    print(f"  L{l}H{h}: {my_head_results[l, h]:.3f}")

### Exercise 2: Find Another Circuit

Apply activation patching to a different task: the "greater-than" task. Use prompts like "The war lasted from 1732 to 17" where the model should predict a number greater than 32. Which layers are critical for this computation?

Create clean prompts with a specific year (e.g., 1732) and corrupted prompts with a different year (e.g., 1501). Measure whether the model predicts valid years (> source year's last two digits) by comparing logits for valid vs. invalid completions.

<details>
<summary>Hint</summary>

Create clean prompts like "The war lasted from 1732 to 17" and corrupted prompts like "The war lasted from 1501 to 17". The logit difference metric should be: sum of logits for valid two-digit completions (33-99) minus sum of logits for invalid ones (00-32). Patch each layer's residual stream from clean into corrupted and measure how much the "valid year" signal recovers. Use `model.to_tokens("33")` etc. to get token IDs for the year digits.

</details>

In [ ]:
# Exercise 2: Find the Greater-Than Circuit

# 1. Define prompts
# TODO: Try other year pairs to see how the circuit generalizes!
clean_prompt_gt = "The war lasted from 1732 to 17"
corrupted_prompt_gt = "The war lasted from 1501 to 17"

# 2. Run both prompts and cache activations
clean_logits_gt, clean_cache_gt = model.run_with_cache(clean_prompt_gt)
corrupted_logits_gt, corrupted_cache_gt = model.run_with_cache(corrupted_prompt_gt)

# 3. Define the metric: sum of logits for valid years minus invalid years
#    For clean (1732): valid completions are 33-99, invalid are 00-32
#    Get token IDs for two-digit numbers
valid_tokens = [model.to_single_token(str(i)) for i in range(33, 100)]
invalid_tokens = [model.to_single_token(str(i).zfill(2)) for i in range(0, 33)]

def greater_than_metric(logits):
    """Positive when model prefers valid (greater) years."""
    final_logits = logits[0, -1]
    valid_sum = final_logits[valid_tokens].sum()
    invalid_sum = final_logits[invalid_tokens].sum()
    return (valid_sum - invalid_sum).item()

# 4. Patch each layer's residual stream from clean into corrupted
layer_results_gt = []
for layer in range(model.cfg.n_layers):
    def patch_resid(activation, hook, l=layer):
        activation[0] = clean_cache_gt[f"blocks.{l}.hook_resid_post"][0]
        return activation
    patched_logits = model.run_with_hooks(
        corrupted_prompt_gt,
        fwd_hooks=[(f"blocks.{layer}.hook_resid_post", patch_resid)]
    )
    layer_results_gt.append(greater_than_metric(patched_logits))

# 5. Plot results
clean_metric = greater_than_metric(clean_logits_gt)
corrupted_metric = greater_than_metric(corrupted_logits_gt)
normalized_gt = [(r - corrupted_metric) / (clean_metric - corrupted_metric + 1e-8) for r in layer_results_gt]

plt.figure(figsize=(12, 5))
plt.bar(range(model.cfg.n_layers), normalized_gt)
plt.xlabel("Layer")
plt.ylabel("Fraction of greater-than signal recovered")
plt.title("Greater-Than Circuit: Layer-by-Layer Activation Patching")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Clean metric: {clean_metric:.2f}")
print(f"Corrupted metric: {corrupted_metric:.2f}")
print(f"Most important layers: {sorted(range(len(normalized_gt)), key=lambda i: normalized_gt[i], reverse=True)[:3]}")

## Section 6: Key Takeaways & Further Reading

**What you should remember:**
- Activation patching is the gold standard for causal claims about model components
- Attribution patching scales this to larger models via gradient approximation
- The IOI task reveals a complex, multi-component circuit with redundancy
- Circuits have functional roles: movers, inhibitors, detectors

**Watch out for:**
- Patching only tests one component at a time (interactions are missed)
- The corrupted input matters -- different corruptions test different things
- Circuits can be fragile to the exact metric used

**Further reading:**
- [Interpretability in the Wild](https://arxiv.org/abs/2211.00593) (Wang et al., 2022) -- the IOI paper
- [Attribution Patching](https://www.neelnanda.io/mechanistic-interpretability/attribution-patching) (Neel Nanda)
- [ACDC: Automatic Circuit Discovery](https://arxiv.org/abs/2304.14997) (Conmy et al., 2023)
- [Towards Best Practices of Activation Patching](https://arxiv.org/abs/2309.16042)

**Next**: Notebook 05 -- Logit Lens (peeking at intermediate predictions)